# Quantum classification

In this notebook, we see the dependence of trainability of the QML classifier on the circuit structure and the cost function.

The circuit structure is defined by two parts: the embedding part, the ansatz part.
- embedding part: the embedding circuit to encode input data into the quantum circuit.  
    You can use 6 types of embedding circuits:
    - Tensor Product Embedding (TPE)
    - Alternating Layered Embedding (ALE)
    - Hardware Efficient Embedding (HEE)
    - Classically Hard Embedding (CHE)
    - Matrix Product State Embedding (MPS)
    - Amplitude Embedding (APE)
- ansatz part: the parametrized circuit to learn the training dataset.  
    You can use 3 types of ansatz circuits:
    - Tensor Product Ansatz (TPA): the ansatz circuit is the tensor product of rotation gates (Input data as angles).
    - Hardware Efficient Ansatz (HEA): the ansatz circuit is the tensor product of rotation gates (Input data as angles) followed by controlled-NOT gates over adjacent qubits.
    - Strongly Entangling Ansatz (SEA): the ansatz circuit that is strongly entangled.

On `HEE` and `CHE`, refer to [`Subtleties in the trainability of quantum machine learning models`](https://arxiv.org/abs/2110.14753) for more details.

On `SEA`, refer to PennyLnae page: [embedding and ansatz](https://pennylane.readthedocs.io/en/stable/introduction/templates.html)

In `quantum_classification_.py`, we adopted local cost function of the form $C_{\mathrm{L}}=\operatorname{Tr}\left[\rho(\boldsymbol{\theta})O_{\mathrm{L}}\right] = 1 - \frac1n\sum_j p_{|{0}\rangle_j}$
, where $ \rho(\boldsymbol{\theta}) = V(\boldsymbol{\theta})|\mathbf{0}\rangle\langle\mathbf{0}| V(\boldsymbol{\theta})^{\dagger}$ and $O_{\mathrm{L}}=\mathbb{1}-\frac{1}{n} \sum_{j=1}^{n}|0\rangle\langle 0|_{j} \otimes \mathbb{1}_{\bar{j}}$


as used in `Cost-Function-Dependent Barren Plateaus in Shallow Quantum Neural Networks`

## 1. Iris dataset with 2 features and 2 labels

In [ ]:
from pennylane import numpy as np
from sklearn.model_selection import train_test_split


import sys
sys.path.append('..')
from src.quantum_classifier_ import QuantumClassifier_

In [ ]:
data = np.loadtxt("../data/iris_classes1and2_scaled.txt")
X = data[:,:2] # use first 2 features out of 4
Y = data[:,-1] # last column is the class
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [ ]:
print(X.shape, Y.shape)
print(x_train.shape, y_train.shape)
print(x_test.shape, y_test.shape)

## TPE TPA (MAE)

In [ ]:
# settings
# nqubits = X.shape[1]
nqubits = 4
embedding_nlayers = 1
ansatz_nlayers = 5
embedding_type = "TPE"
ansatz_type = "HEA"
cost_type = "MAE"
label = f"{embedding_type}, {ansatz_type}"

iris_tpe_tpa_mae = QuantumClassifier_(
    x_train,
    y_train,
    nqubits,
    embedding_nlayers,
    ansatz_nlayers,
    embedding_type,
    ansatz_type,
    cost_type,
    shots=None,
    stepsize=0.3,
    steps=50,
)

In [ ]:
iris_tpe_tpa_mae.draw_circuit(decompose=True)

In [ ]:
iris_tpe_tpa_mae.optimize()

In [ ]:
iris_tpe_tpa_mae.plot_cost()

In [ ]:
print('accuracy ', iris_tpe_tpa_mae.accuracy(x_test, y_test))
print('optimized cost; ', iris_tpe_tpa_mae.cost_list[-1])

In [ ]:
from matplotlib import pyplot as plt

l = np.array(iris_tpe_tpa_mae.diff_list).T

for i in range(len(l)):
    plt.plot(np.arange(50), l[i])

plt.plot(np.arange(50), np.ones(50)*0.5, 'k--')

plt.xlabel("steps", fontsize=18)
plt.ylabel("$|\ell_i({\\bf{\\theta}}) - y_i|$", fontsize=18)
plt.ylim(0, 1)
plt.title("$|\ell_i({\\bf{\\theta}}) - y_i|$ for all training data (MAE)", fontsize=18)
plt.show()

In [ ]:
steps = 50

plt.plot(np.arange(steps), np.sum(np.array(iris_tpe_tpa_mae.diff_list)/80, axis=1))
plt.xlabel("steps", fontsize=18)
plt.ylabel("MAE", fontsize=18)
plt.title("MAE for each step parameters", fontsize=18)
plt.show()

## TPE TPA (MSE)

In [ ]:
# settings
# nqubits = X.shape[1]
nqubits = 4
embedding_nlayers = 1
ansatz_nlayers = 5
embedding_type = "TPE"
ansatz_type = "HEA"
cost_type = "MSE"
label = f"{embedding_type}, {ansatz_type}"

iris_tpe_tpa_mse = QuantumClassifier_(
    x_train,
    y_train,
    nqubits,
    embedding_nlayers,
    ansatz_nlayers,
    embedding_type,
    ansatz_type,
    cost_type,
    shots=None,
    stepsize=0.3,
    steps=50,
)

In [ ]:
iris_tpe_tpa_mse.draw_circuit(decompose=True)

In [ ]:
iris_tpe_tpa_mse.optimize()

In [ ]:
iris_tpe_tpa_mse.plot_cost()

In [ ]:
print('accuracy ', iris_tpe_tpa_mse.accuracy(x_test, y_test))
print('optimized cost; ', iris_tpe_tpa_mse.cost_list[-1])

In [ ]:
from matplotlib import pyplot as plt

l = np.array(iris_tpe_tpa_mse.diff_list).T

for i in range(len(l)):
    plt.plot(np.arange(50), l[i])

plt.plot(np.arange(50), np.ones(50)*0.5, 'k--')

plt.xlabel("steps", fontsize=18)
plt.ylabel("$|\ell_i({\\bf{\\theta}}) - y_i|$", fontsize=18)
plt.ylim(0, 1)
plt.title("$|\ell_i({\\bf{\\theta}}) - y_i|$ for all training data (MSE)", fontsize=18)
plt.show()

In [ ]:
steps = 50

plt.plot(np.arange(steps), np.sum(np.array(iris_tpe_tpa_mse.diff_list)/80, axis=1))
plt.xlabel("steps", fontsize=18)
plt.ylabel("MAE", fontsize=18)
plt.title("MAE for each step parameters", fontsize=18)
plt.show()

## TPE TPA (LOG)

In [ ]:
# settings
# nqubits = X.shape[1]
nqubits = 4
embedding_nlayers = 1
ansatz_nlayers = 5
embedding_type = "TPE"
ansatz_type = "HEA"
cost_type = "LOG"
label = f"{embedding_type}, {ansatz_type}"

iris_tpe_tpa_log = QuantumClassifier_(
    x_train,
    y_train,
    nqubits,
    embedding_nlayers,
    ansatz_nlayers,
    embedding_type,
    ansatz_type,
    cost_type,
    shots=None,
    stepsize=0.3,
    steps=50,
)

In [ ]:
iris_tpe_tpa_log.draw_circuit(decompose=True)

In [ ]:
iris_tpe_tpa_log.optimize()

In [ ]:
iris_tpe_tpa_log.plot_cost()

In [ ]:
print('accuracy ', iris_tpe_tpa_log.accuracy(x_test, y_test))
print('optimized cost; ', iris_tpe_tpa_log.cost_list[-1])

In [ ]:
from matplotlib import pyplot as plt

l = np.array(iris_tpe_tpa_log.diff_list).T

for i in range(len(l)):
    plt.plot(np.arange(50), l[i])

plt.plot(np.arange(50), np.ones(50)*0.5, 'k--')

plt.xlabel("steps", fontsize=18)
plt.ylabel("$|\ell_i({\\bf{\\theta}}) - y_i|$", fontsize=18)
plt.ylim(0, 1)
plt.title("$|\ell_i({\\bf{\\theta}}) - y_i|$ for all training data (LOG)", fontsize=18)
plt.show()

In [ ]:
steps = 50

plt.plot(np.arange(steps), np.sum(np.array(iris_tpe_tpa_log.diff_list)/80, axis=1))
plt.xlabel("steps", fontsize=18)
plt.ylabel("MAE", fontsize=18)
plt.title("MAE for each step parameters", fontsize=18)
plt.show()

In [2]:
import numpy as np
spectrum = np.exp(np.linspace(np.log(0.001), np.log(1), 10))
print(spectrum)

[0.001      0.00215443 0.00464159 0.01       0.02154435 0.04641589
 0.1        0.21544347 0.46415888 1.        ]
